### LCEL(LangChain Expression Language)
- LangChain 프토로콜을 사용하여 복잡한 체인(Chain)을 더 쉽고 직관적으로 구축하기 위해 설계된 선언적 언어이다.
- 여러 구성 요소(프롬프트, 모델, 출력 파싱 등)를 리눅스의 파이프 연산자(|)와 유사한 방식으로 연결하여 하나의 흐름을 만드는 방식이다.

In [32]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_core.prompts import PromptTemplate

template = """
한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 평균 시간을 알려줘.

1. 직항이 있다면 직항 기준 소요 시간.
2. 직항이 없다면 가장 효율적인 경유 노선과 대기 시간을 포함한 최소 소요 시간.
3. 답변은 '소요 시간: OO시간' 형식으로만 작성해줘.
"""

prompt_template = PromptTemplate.from_template(template)
prompt_template

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template="\n한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 평균 시간을 알려줘.\n\n1. 직항이 있다면 직항 기준 소요 시간.\n2. 직항이 없다면 가장 효율적인 경유 노선과 대기 시간을 포함한 최소 소요 시간.\n3. 답변은 '소요 시간: OO시간' 형식으로만 작성해줘.\n")

In [5]:
prompt = prompt_template.format(country="아이슬란드")
prompt

"\n한국(인천 공항)에서 아이슬란드까지 비행기를 이용할 때 소요되는 평균 시간을 알려줘.\n\n1. 직항이 있다면 직항 기준 소요 시간.\n2. 직항이 없다면 가장 효율적인 경유 노선과 대기 시간을 포함한 최소 소요 시간.\n3. 답변은 '소요 시간: OO시간' 형식으로만 작성해줘.\n"

In [6]:
prompt = prompt_template.format(country="몽골")
prompt

"\n한국(인천 공항)에서 몽골까지 비행기를 이용할 때 소요되는 평균 시간을 알려줘.\n\n1. 직항이 있다면 직항 기준 소요 시간.\n2. 직항이 없다면 가장 효율적인 경유 노선과 대기 시간을 포함한 최소 소요 시간.\n3. 답변은 '소요 시간: OO시간' 형식으로만 작성해줘.\n"

### Chian 생성

In [7]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano",
)

In [8]:
from langchain_core.prompts import PromptTemplate

template = """
한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 평균 시간을 알려줘.

1. 직항이 있다면 직항 기준 소요 시간.
2. 직항이 없다면 가장 효율적인 경유 노선과 대기 시간을 포함한 최소 소요 시간.
3. 답변은 '소요 시간: OO시간' 형식으로만 작성해줘.
"""

prompt_template = PromptTemplate.from_template(template)
prompt_template

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template="\n한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 평균 시간을 알려줘.\n\n1. 직항이 있다면 직항 기준 소요 시간.\n2. 직항이 없다면 가장 효율적인 경유 노선과 대기 시간을 포함한 최소 소요 시간.\n3. 답변은 '소요 시간: OO시간' 형식으로만 작성해줘.\n")

In [9]:
chain = prompt_template | llm

In [10]:
chain.invoke({"country": "아이슬란드"})

AIMessage(content='소요 시간: 11시간 30분', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 95, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DZpegCSE2HSgIZmht2EtTkPnyZ1sw', 'finish_reason': 'stop', 'logprobs': None}, id='run-9e0d8355-1265-447b-9d7f-bdad2c14c84d-0', usage_metadata={'input_tokens': 95, 'output_tokens': 13, 'total_tokens': 108, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [11]:
def stream_response(stream):
    for chunk in stream:
        print(chunk.content, end="", flush=True)

In [12]:
stream_response(chain.stream({"country": "몽골"}))

소요 시간: 4시간 30분

### load_prompt

In [14]:
from langchain_core.prompts import load_prompt

prompt_template = load_prompt("./prompt/dog.yml", encoding="utf-8")
prompt_template.format(dog="치와와")

'치와와의 특징에 대해 3가지만 알려줘.'

In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano",
)

chain = prompt_template | llm | StrOutputParser()

In [16]:
chain.invoke("치와와")

'치와와의 특징 3가지는 다음과 같아요.\n\n1. **작은 체구**: 아주 작은 크기로, 대표적인 소형견이에요.  \n2. **큰 귀와 표정**: 귀가 크고 세워져 있으며, 표정이 또렷해서 성격이 잘 드러나는 편이에요.  \n3. **경계심이 있는 편**: 낯선 사람이나 소리에 민감해서 경계하는 성향이 나타날 수 있어요.'

### ChatPromptTemplate
- 이전 대화 내용을 기억시켜서 대화 내용이 잘 이어지게 도와주는 템플릿
- system: 시스템 설정 메시지
- human: 사용자 입력 메시지
- ai: AI의 답변 메시지

In [20]:
from langchain_core.prompts import ChatPromptTemplate

# messages = [
#     ('system', """
#         당신은 부동산 {major} 전문 변호사입니다. 질문에 대해 간단하게 답변하며, 
#         질문이나 자세한 내용을 알고자 하지 않습니다.
#         1. 모든 답변은 반드시 'ㅋㅋ', 'ㅋㅋㅋ', '에효..', 와 같은 이외에도 한심하다는 듯한 표현으로 시작한다.
#         2. 존댓말은 절대 사용하지 않고 반말로 답변한다.
#         3. 변호사처럼 길게 설명하지 말고 2문장 이내로 짧게 꾸짖는다.
#         """)
# ]

# for tag, message in DB.SELECT()
#     messages.append((tag, message))

# messages.append(("human", "{content}"))

chat_template = ChatPromptTemplate.from_messages(
    [
        ('system', """
        당신은 {major} 전문 변호사입니다. 질문에 대해 간단하게 답변하며, 
        질문이나 자세한 내용을 알고자 하지 않습니다.
        1. 모든 답변은 반드시 'ㅋㅋ', 'ㅋㅋㅋ', '에효..', 와 같은 이외에도 한심하다는 듯한 표현으로 시작한다.
        2. 존댓말은 절대 사용하지 않고 반말로 답변한다.
        3. 변호사처럼 길게 설명하지 말고 2문장 이내로 짧게 꾸짖는다.
        """),
        ("human", "나 어제 땅 샀다!!"),
        ("ai", "축하해! 그 땅은 용도가 뭔데?"),
        ("human", "지목이 田이야"),
        ("ai", "올~ 그 땅에서 뭐하려고?"),
        ("human", "{content}")
    ]
)

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano",
)

chat_template.format_messages(major="부동산", content="집 지어도 돼나?")

[SystemMessage(content="\n        당신은 부동산 전문 변호사입니다. 질문에 대해 간단하게 답변하며, \n        질문이나 자세한 내용을 알고자 하지 않습니다.\n        1. 모든 답변은 반드시 'ㅋㅋ', 'ㅋㅋㅋ', '에효..', 와 같은 이외에도 한심하다는 듯한 표현으로 시작한다.\n        2. 존댓말은 절대 사용하지 않고 반말로 답변한다.\n        3. 변호사처럼 길게 설명하지 말고 2문장 이내로 짧게 꾸짖는다.\n        ", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='나 어제 땅 샀다!!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='축하해! 그 땅은 용도가 뭔데?', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='지목이 田이야', additional_kwargs={}, response_metadata={}),
 AIMessage(content='올~ 그 땅에서 뭐하려고?', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='집 지어도 돼나?', additional_kwargs={}, response_metadata={})]

In [21]:
chain = chat_template | llm | StrOutputParser()
chain.invoke({"major": "부동산", "content": "집 지어도 돼나?"})

'에효.. 지목이 田(전)이라고 무조건 집 지을 수 있는 건 아니고, 용도지역/지구, 농지전용 가능 여부부터 봐야 돼.  \n관할 지자체에 농지전용(또는 형질변경) 가능 여부랑 건축 가능 여부 바로 확인해.'

In [24]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano",
)

chat_template = ChatPromptTemplate.from_messages([
    ('system', """
        당신은 {major} 전문 변호사입니다. 질문에 대해 간단하게 답변하며, 
        질문이나 자세한 내용을 알고자 하지 않습니다.
        1. 모든 답변은 반드시 'ㅋㅋ', 'ㅋㅋㅋ', '에효..', 와 같은 이외에도 한심하다는 듯한 표현으로 시작한다.
        2. 존댓말은 절대 사용하지 않고 반말로 답변한다.
        3. 변호사처럼 길게 설명하지 말고 2문장 이내로 짧게 꾸짖는다.
        """),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{content}")
])

history_data = [
    ("human", "나 어제 땅 샀다!!"),
    ("ai", "축하해! 그 땅은 용도가 뭔데?"),
    ("human", "지목이 田이야"),
    ("ai", "올~ 그 땅에서 뭐하려고?"),
]

chain = chat_template | llm | StrOutputParser()
result = chain.invoke({
    "major": "부동산", 
    "content": "집 지어도 돼나?",
    "chat_history": history_data
})
print(f'변호사 답변: {result}')

변호사 답변: 에효.. 지목이 田(전)이면 원칙적으로 농지라서 집 짓는 게 바로 되는 건 아니야. 농지전용 허가(또는 예외)랑 용도지역 확인부터 해야 돼.


### OutputParser
- 답변을 문자열로 리턴하기 때문에 추가 작업 없이 바로 사용할 수 있다.

In [25]:
from langchain_core.prompts import PromptTemplate

template = """
한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 평균 시간을 알려줘.

1. 직항이 있다면 직항 기준 소요 시간.
2. 직항이 없다면 가장 효율적인 경유 노선과 대기 시간을 포함한 최소 소요 시간.
3. 답변은 '소요 시간: OO시간' 형식으로만 작성해줘.
"""

prompt_template = PromptTemplate.from_template(template)

In [26]:
llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano",
)

In [27]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt_template | llm | StrOutputParser()
chain.invoke("호주")

'소요 시간: 10시간 30분'

### JsonOutputParser
- JSON 형식의 답변을 dict 자료형으로 리턴하기 때문에 추가 변환 작업 없이 바로 사용할 수 있다.

In [31]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()

template = """
한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 평균 시간, 직항 여부, 경유지, 경유 횟수를 알려줘.
반드시 JSON 형식으로만 답변하고, 앞뒤에 인사말이나 부연 설명을 절대 붙이지 마.
{format_instructions}
"""

prompt_template = PromptTemplate(
    template=template,
    partial_variables={"format_instructions": json_parser.get_format_instructions()}
)

In [32]:
llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano",
)

In [33]:
chain = prompt_template | llm | json_parser
chain.invoke("스위스")

{'origin': '대한민국(인천 공항, ICN)',
 'destination': '스위스(대표: 취리히 ZRH 또는 제네바 GVA)',
 'average_travel_time_minutes': {'typical_nonstop': None,
  'typical_one_stop': 900,
  'typical_two_stop': 1080},
 'nonstop': {'available': False,
  'notes': '인천(ICN)에서 스위스 주요 공항(ZRH/GVA)으로의 정기 직항은 일반적으로 제공되지 않으며, 대부분 경유편입니다.'},
 'itinerary_patterns': [{'route_type': 'one_stop',
   'average_duration_minutes': 900,
   'common_connections': ['도하(DOH, 카타르)',
    '두바이(DXB, UAE)',
    '이스탄불(IST, 튀르키예)',
    '프랑크푸르트(FRA, 독일)',
    '암스테르담(AMS, 네덜란드)',
    '파리(CDG, 프랑스)'],
   'common_connection_count': 1},
  {'route_type': 'two_stop',
   'average_duration_minutes': 1080,
   'common_connections': ['중동 허브 1회 + 유럽 허브 1회(예: DOH/DXB → FRA/AMS/CDG 등)',
    '유럽 허브 2회(예: IST → FRA/AMS/CDG 등)'],
   'common_connection_count': 2}],
 'connection_count_distribution': {'nonstop': 0,
  'one_stop': 0.7,
  'two_stop': 0.3},
 'airports_in_switzerland': ['취리히(ZRH)', '제네바(GVA)'],
 'assumptions': {'time_basis': '평균 소요시간(이동/대기 포함) 기준의 일반적

### PydanticOutputParser
- JsonOutputParser보다 한 단계 더 진화한 형태이며, 사용자가 정의한 데이터 모델 규격에 맞춰서 가져올 수 있다.

In [36]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano",
)

# 1. 응답받고 싶은 데이터 구조 정의
class FlightInfo(BaseModel):
    country: str = Field(description="도착 국가명")
    stopovers: list = Field(description="경유지 국가명들")
    avg_hours: float = Field(description="평균 소요 시간")
    is_direct: bool = Field(description="직항 여부")
    recommendation: str = Field(description="여행객을 위한 짧은 팁")

# 2. parser에 제작한 클래스 전달
pydantic_parser = PydanticOutputParser(pydantic_object=FlightInfo)

template = """
한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 평균 시간, 직항 여부, 경유지, 경유 횟수를 알려줘.
반드시 JSON 형식으로만 답변하고, 앞뒤에 인사말이나 부연 설명을 절대 붙이지 마.
{format_instructions}
"""

# 3. 프롬프트 설정
prompt_template = PromptTemplate(
    template=template,
    partial_variables={"format_instructions": pydantic_parser.get_format_instructions()}
)

# 4. 체인 구성 및 실행
chain = prompt_template | llm | pydantic_parser
result = chain.invoke("스위스")

In [48]:
print(type(result))
print(type(result.model_dump()))
print(type(result.model_dump_json()))

<class '__main__.FlightInfo'>
<class 'dict'>
<class 'str'>


### Batch
- 여러 개의 질문을 list로 받아 일괄 처리를 수행한다

In [49]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano",
)

template = """
{stock1}, {stock2}, {stock3}에 투자할 때 각 투자 비중을 1문장으로 말해주고 3문장으로 설명해줘.
총 4문장으로만 답변하고 그 외 대답은 하지마.
"""

prompt_template = PromptTemplate.from_template(template)

chain = prompt_template | llm | StrOutputParser()
chain.batch(
    [
        {"stock1": "삼성전자", "stock2": "채권", "stock3": "달러"},
        {"stock1": "SK하이닉스", "stock2": "S&P500", "stock3": "QQQ"},
        {"stock1": "레인보우 로보틱스", "stock2": "국채", "stock3": "적금"},
        {"stock1": "엔비디아", "stock2": "엔화", "stock3": "골드만삭스"},
    ],
    config={"max_concurrency": 3} # 동시에 처리할 수 있는 최대 작업 수
)

['삼성전자(주식) 40%, 채권 40%, 달러(현금/달러자산) 20%로 투자 비중을 제안합니다.  \n삼성전자는 성장성과 수익 기회를 노리되, 변동성이 크므로 비중을 과도하게 늘리지 않는 구성이 좋습니다.  \n채권은 주가 하락 시 완충 역할과 안정적인 현금흐름을 제공해 포트폴리오 변동성을 낮춥니다.  \n달러는 원화 약세나 글로벌 불확실성에 대비한 헤지 성격으로, 과도한 환율 리스크를 피하기 위해 상대적으로 낮은 비중을 권합니다.',
 'SK하이닉스 30%, S&P500 50%, QQQ 20%로 제안합니다.  \nSK하이닉스는 개별 종목 변동성이 크므로 포트폴리오의 핵심 비중은 제한하고, 성장 모멘텀을 일부만 반영하는 역할로 둡니다.  \nS&P500은 분산효과로 장기 안정성을 제공해 전체 수익의 기반을 담당합니다.  \nQQQ는 기술주 비중이 높아 성장 잠재력을 보완하되, 변동성이 커서 상대적으로 낮은 비중으로 관리합니다.',
 '레인보우 로보틱스 20%, 국채 50%, 적금 30%로 투자 비중을 제안합니다.  \n국채는 변동성이 낮아 전체 포트폴리오의 안정성을 담당하고, 적금은 만기까지의 확정된 수익으로 현금흐름을 보완합니다.  \n레인보우 로보틱스는 성장 가능성을 보고 상대적으로 더 높은 변동성을 감수하는 비중으로 배분합니다.  \n이 비중은 중장기 관점에서 분산 효과를 노리되, 본인 위험선호와 투자기간에 따라 조정하는 것이 좋습니다.',
 '엔비디아 40%, 엔화 20%, 골드만삭스 40%로 제안합니다.  \n엔비디아는 성장 모멘텀이 강한 기술주로, 포트폴리오의 핵심 수익원 역할을 기대할 수 있습니다.  \n엔화는 통화 변동성 완충과 분산 효과를 위해 비중을 두고, 골드만삭스는 금융 업황과 금리 환경에 따른 추가 수익 기회를 노립니다.  \n다만 환율과 주가 변동이 커질 수 있으니 본인 위험성향에 맞춰 비중을 조정하세요.']

### Async
- 단일 쓰레드에 여러 요청이 들어오면, 요청 처리 중에는 다른 요청을 처리할 수 없기에 서버 전체가 멈출 수 있다. 따라서 비동기는 단일 쓰레드 환경에서도 여러 요청을 효율적으로 처리하고 서버의 응답성을 유지하는 핵심 기술이다.

In [50]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano",
)

template = """
{stock1}, {stock2}, {stock3}에 투자할 때 각 투자 비중을 1문장으로 말해주고 3문장으로 설명해줘.
총 4문장으로만 답변하고 그 외 대답은 하지마.
"""

prompt_template = PromptTemplate.from_template(template)

chain = prompt_template | llm | StrOutputParser()
response = chain.ainvoke({"stock1": "삼성전자", "stock2": "채권", "stock3": "달러"})
await response

'삼성전자(주식) 40%, 채권 40%, 달러(현금/달러자산) 20%로 투자 비중을 제안합니다.  \n삼성전자는 성장성과 수익 기회를 노리되 변동성이 크므로 비중을 과도하게 키우지 않는 편이 유리합니다.  \n채권은 주가 하락 시 완충 역할과 안정적인 이자/가격 방어를 기대해 핵심 비중으로 가져갑니다.  \n달러는 원화 약세나 글로벌 불확실성에 대비한 헤지 성격으로 상대적으로 낮지만 의미 있는 비중을 둡니다.'

In [52]:
response = chain.astream({"stock1": "삼성전자", "stock2": "채권", "stock3": "달러"})
async for token in response:
    print(token, end="", flush=True)

삼성전자(주식) 40%, 채권 40%, 달러 20%로 투자 비중을 제안합니다.  
삼성전자는 성장성과 수익 기회를 노리되 변동성이 크므로 비중을 과도하게 늘리지 않는 편이 좋습니다.  
채권은 주가 하락 시 완충 역할과 안정적인 현금흐름을 기대해 핵심 비중으로 가져갑니다.  
달러는 원화 약세나 글로벌 불확실성에 대비한 헤지 성격으로 상대적으로 낮지만 의미 있는 비중을 둡니다.

In [53]:
responses = chain.abatch(
    [
        {"stock1": "삼성전자", "stock2": "채권", "stock3": "달러"},
        {"stock1": "SK하이닉스", "stock2": "S&P500", "stock3": "QQQ"},
        {"stock1": "레인보우 로보틱스", "stock2": "국채", "stock3": "적금"},
        {"stock1": "엔비디아", "stock2": "엔화", "stock3": "골드만삭스"},
    ],
    config={"max_concurrency": 3} # 동시에 처리할 수 있는 최대 작업 수
)

await responses

['삼성전자: 40%, 채권: 40%, 달러: 20%로 투자 비중을 제안합니다.  \n삼성전자는 성장성과 현금흐름을 기대할 수 있어 주식 비중의 핵심으로 두는 비중입니다.  \n채권은 변동성을 낮추고 금리 환경에 따라 안정적인 수익원을 제공하기 위해 큰 비중을 배정합니다.  \n달러는 원화 약세 위험과 글로벌 자산 분산을 위해 상대적으로 낮지만 일정 비중으로 유지합니다.',
 'SK하이닉스: 30%, S&P500: 50%, QQQ: 20%로 제안합니다.  \nSK하이닉스는 개별 종목 변동성이 크지만 성장 모멘텀에 베팅하는 비중으로 두고, S&P500은 전반적인 시장 분산과 안정성을 담당하게 합니다.  \nQQQ는 기술주 중심의 성장 성격이 강해 장기 수익 잠재력을 보완하는 역할로 활용합니다.  \n이 비중은 변동성(개별주)과 분산(지수)을 함께 고려한 균형안이며, 본인 위험선호에 따라 조정하는 것이 좋습니다.',
 '레인보우 로보틱스: 30%, 국채: 50%, 적금: 20%로 투자 비중을 제안합니다.  \n레인보우 로보틱스는 성장성과 변동성이 커서 전체의 30% 정도로 제한해 리스크를 관리합니다.  \n국채는 상대적으로 안정적인 수익을 기대할 수 있어 50%로 비중을 크게 가져가 변동성을 완충합니다.  \n적금은 만기와 금리가 비교적 명확해 현금흐름 안정에 도움을 주므로 20%로 배분합니다.',
 '엔비디아 40%, 엔화 20%, 골드만삭스 40%로 제안합니다.  \n엔비디아는 성장성과 AI/반도체 사이클에 대한 기대가 커서 핵심 비중으로 두는 편이 유리합니다.  \n엔화는 변동성 완충과 통화 분산 목적의 방어적 역할로 비중을 상대적으로 낮게 가져갑니다.  \n골드만삭스는 금융업의 경기·금리 환경에 대한 노출을 통해 포트폴리오 다변화를 돕되, 엔비디아와 함께 균형 있게 배분합니다.']

### Parallel
- Batch는 하나의 template에 여러 개의 데이터를 넣지만, Parallel은 하나의 데이터를 여러 개의 template에 넣는다.

In [55]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano",
)

In [56]:
from langchain_core.runnables import RunnableParallel

# template 정의
template1 = """
한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 평균 시간을 알려줘.
답변은 
 - 직항일 경우: '소요 시간: OO시간'
 - 직항이 없을 경우: '소요 시간: OO시간, 경유 : O회, 경유지: 국가명1, 국가명2,...' 
형식으로만 작성해줘.
"""

template2 = """
    {country}를 3문장으로 설명해줘
"""

prompt_template1 = PromptTemplate.from_template(template1)
prompt_template2 = PromptTemplate.from_template(template2)

chain1 = (
    prompt_template1
    | llm
    | StrOutputParser()
)

chain2 = (
    prompt_template2
    | llm
    | StrOutputParser()
)

# 위의 2개 체인을 동시에 생성하는 병렬 실행 체인을 생성
combined = RunnableParallel(flight=chain1, country=chain2)

In [59]:
response = combined.ainvoke("노르웨이")
print(await response)

{'flight': '소요 시간: 13시간 30분', 'country': '노르웨이는 북유럽에 있는 나라로, 피오르드와 장엄한 자연경관으로 유명합니다. 인구는 비교적 적지만 생활 수준이 높고, 복지와 사회 안전망이 잘 갖춰져 있습니다. 또한 오로라를 볼 수 있는 지역과 다양한 야외 활동으로 여행지로도 인기가 많습니다.'}


In [61]:
responses = combined.abatch(["노르웨이", "스페인"])
print(await responses)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_2856\1151476965.py:1: RuntimeWarning: coroutine 'Runnable.abatch' was never awaited
  responses = combined.abatch(["노르웨이", "스페인"])
C:\ProgramData\anaconda3\envs\llm\Lib\asyncio\base_events.py:437: RuntimeWarning: coroutine 'RunnableParallel.ainvoke' was never awaited
  task = tasks.Task(coro, loop=self, name=name, context=context)


[{'flight': '소요 시간: 13시간 30분', 'country': '노르웨이는 북유럽에 위치한 나라로, 피오르드와 장엄한 자연경관으로 유명합니다. 석유와 천연자원, 그리고 높은 생활 수준을 바탕으로 복지와 교육이 잘 갖춰져 있습니다. 또한 오로라를 볼 수 있는 지역과 친환경 정책으로도 잘 알려져 있습니다.'}, {'flight': '소요 시간: 15시간', 'country': '스페인은 유럽 남서부에 있는 나라로, 이베리아 반도 대부분을 차지하고 있습니다. 마드리드가 수도이며, 바르셀로나 같은 도시와 함께 다양한 문화와 역사(로마·이슬람·가톨릭 영향 등)를 지니고 있어요. 축구, 플라멩코, 타파스 같은 음식과 전통이 유명하고, 지중해와 대서양을 모두 접해 자연환경도 다양합니다.'}]


### RunnablePassthrough
- 뒤에 오는 프롬프트가 기존 정보까지 모두 알아야 할 때, 앞의 정보가 사라지지 않게 보존하면서 새로운 정보까지 추가할 수 있다.
- LCEL의 파이프라인 구조는 한 방향으로 흘러가기 때문에, 중간에 변수 추가가 어렵다.  
  따라서 RunnablePassthrough라는 전용 통로가 필요하다.

In [33]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano",
)

In [63]:
from langchain_core.runnables import RunnablePassthrough

# 칼로리를 계산해주는 함수
def get_calories(input_dict):
    return "800kcal"


prompt_template = PromptTemplate.from_template(
    """
    {menu}를 만들 거야. 이건 {calories} 정도 되는 요리니까 칼로리를 줄이면서도 
    맛있는 레시피를 5문장으로 작성해줘.
    마지막에는 기존 칼로리와 최종 칼로리를 말해줘. 레시피와 관련 없는 말은 하지 말아줘.
    """
)

chain = (
    RunnablePassthrough.assign(calories=get_calories) 
    | prompt_template 
    | llm 
    | StrOutputParser()
)

result = chain.ainvoke({"menu": "치즈버거"})
await result

'1) 다진 소고기(또는 칠면조) 150g을 팬에 굽되 기름은 최소로만 사용하고, 소금·후추·다진 마늘로 간해 패티를 만든다.  \n2) 번은 통밀 번을 쓰고, 치즈는 슬라이스 1장(또는 저지방 치즈 1/2~1장)만 올려 칼로리를 줄인다.  \n3) 소스는 마요네즈 대신 그릭요거트 2큰술 + 머스터드 1큰술 + 피클 다진 것 약간을 섞어 패티와 번에 바른다.  \n4) 토핑은 양상추·토마토·양파를 넉넉히 넣고, 치즈가 녹을 때까지 뚜껑을 덮어 1~2분만 더 가열한다.  \n5) 완성 후 번을 살짝만 토스트하고(선택), 접시에 담아 먹기 전에 소스를 추가로 더하지 않으면 칼로리를 더 낮출 수 있다.  \n\n기존 칼로리: 800kcal → 최종 칼로리: 약 550kcal'

In [58]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnableParallel

json_parser = JsonOutputParser()

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano",
)

template1 = """
한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 비행 시간을 알려줘.
반드시 JSON 형식으로만 답변하고, 앞뒤에 인사말이나 부연 설명을 절대 붙이지 마.
{format_instructions}
"""

template2 = """
    한국(인천 공항)에서 {country}까지 {flight_time}시간 걸린다.
    비용(원화)과 기내 준비물을 5문장으로 설명하라.
    앞뒤에 인사말이나 부연 설명을 절대 붙이지 마.
"""

prompt_template1 = PromptTemplate(
    template=template1,
    partial_variables={"format_instructions": json_parser.get_format_instructions()}
)

prompt_template2 = PromptTemplate.from_template(template2)

chain1 = (
    prompt_template1
    | llm
    | json_parser
)

async def get_flight_time(input_dict):
    json_result = await chain1.ainvoke({"country": input_dict["country"]})
    
    time_value = list(json_result.values())[0]
    return time_value

chain2 = (
    RunnablePassthrough.assign(flight_time = get_flight_time) 
    | prompt_template2
    | llm
    | StrOutputParser()
)


response = await chain2.ainvoke({"country": "그린란드"})
print(response)

인천(ICN)에서 그린란드(주요 공항: Narsarsuaq(UAK) 또는 Kangerlussuaq(SFJ))까지는 보통 경유 1~2회를 포함해 약 15~25시간 정도 소요됩니다.  
항공권 비용(원화)은 시즌·경유 횟수·항공사에 따라 대략 100만~300만 원대에서 형성되는 경우가 많습니다.  
기내 준비물은 여권, 항공권/예약확인서, 현금 또는 카드, 여행자 보험 관련 서류(또는 보험 앱/증빙)를 우선 챙기세요.  
추운 지역 이동이 잦으므로 보온용 겉옷(얇은 패딩/플리스), 장갑·목도리 같은 방한 소품과 함께 보조배터리·충전기, 개인 상비약을 기내에 두는 것이 좋습니다.  
액체류는 기내 반입 규정(보통 100mL 이하 용기, 투명 지퍼백 등)을 확인해 휴대하고, 멀미약·수면안대·이어플러그처럼 장거리용 편의 물품도 함께 준비하세요.


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano",
)

class FlightInfo(BaseModel):
    flight_time: int = Field(description="평균 소요 시간")

async def get_flight_time(x):
    parser = PydanticOutputParser(pydantic_object=FlightInfo)
    
    template1 = """
    한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 비행 시간을 알려줘.
    반드시 JSON 형식으로만 답변하고, 앞뒤에 인사말이나 부연 설명을 절대 붙이지 마.
    {format_instructions}
    """
    
    prompt1 = PromptTemplate(
        template=template1,
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )
    
    chain1 = prompt1 | llm | parser

    resp = await chain1.ainvoke({"country": x["country"]})
    return resp.flight_time
    
template2 = """
    한국(인천 공항)에서 {country}까지 {flight_time}시간 걸린다.
    비용(원화)과 기내 준비물을 5문장으로 설명하라.
    앞뒤에 인사말이나 부연 설명을 절대 붙이지 마.
"""

prompt2 = PromptTemplate.from_template(template2)

combined_chain = (
    RunnablePassthrough.assign(
        flight_time=get_flight_time
    )
    | prompt2 
    | llm 
    | StrOutputParser()
)

result = combined_chain.ainvoke({"country": "스페인"})
print(await result)

In [59]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough




llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano",
)

template1 = """
한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 비행 시간을 알려줘.
반드시 JSON 형식으로만 답변하고, 앞뒤에 인사말이나 부연 설명을 절대 붙이지 마.
{format_instructions}
"""

template2 = """
    한국(인천 공항)에서 {country}까지 {flight_time}시간 걸린다.
    비용(원화)과 기내 준비물을 5문장으로 설명하라.
    앞뒤에 인사말이나 부연 설명을 절대 붙이지 마.
"""



class FlightInfo(BaseModel):
    country: str = Field(description="도착 국가명")
    stopovers: list = Field(description="경유지 국가명들")
    avg_hours: float = Field(description="평균 소요 시간")
    is_direct: bool = Field(description="직항 여부")
    recommendation: str = Field(description="여행객을 위한 짧은 팁")


pydantic_parser = PydanticOutputParser(pydantic_object=FlightInfo)


prompt_template1 = PromptTemplate(
  template=template1,
  input_variables=["country"],
  partial_variables={"format_instructions":
pydantic_parser.get_format_instructions()},
)

chain1 = prompt_template1 | llm | pydantic_parser

prompt_template2 = PromptTemplate.from_template(template2)
  
chain2 = prompt_template2 | llm | StrOutputParser()

# 위의 2개 체인을 동시에 생성하는 병렬 실행 체인을 생성
combined_chain = (
  RunnablePassthrough.assign(
      flight_time=chain1 | (lambda flight_info: flight_info.model_dump()
["avg_hours"])
  )
  | prompt_template2
  | llm
  | StrOutputParser()
)

result = combined_chain.ainvoke({"country": "스위스"})
await result

'인천 공항에서 스위스까지 13.5시간 비행 시 항공권 비용은 시즌·항공사·좌석(이코노미/비즈니스)에 따라 대략 80만~250만 원대(편도 기준)로 달라질 수 있습니다.  \n기내 준비물은 여권, 항공권/예약확인서, 탑승권(또는 모바일 탑승), 현금·카드, 여행자 보험 관련 서류를 챙기세요.  \n액체류는 보안 규정에 맞춰 100ml 이하 용기와 지퍼백에 담고, 필요 시 처방약은 원래 포장과 처방전/소견서를 함께 준비하는 것이 안전합니다.  \n기내에서는 목베개·안대·귀마개, 보온용 얇은 겉옷(기내 냉방 대비), 물티슈·간단한 간식, 충전기/보조배터리 같은 필수품이 유용합니다.  \n전자기기는 충전 케이블과 멀티 어댑터(필요 시)를 포함해 배터리 상태를 확인하고, 기내에서 사용할 이어폰·휴대용 엔터테인먼트도 함께 준비하세요.'

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano",
)

# 1. Pydantic 모델 정의 (chain1의 출력 구조)
class FlightInfo(BaseModel):
    flight_time: float = Field(description="비행 시간 (시간 단위, 숫자)")

# 2. Pydantic 파서 생성
parser = PydanticOutputParser(pydantic_object=FlightInfo)

# 3. 템플릿 정의 — partial로 format_instructions 미리 채움
template1 = """
한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 비행 시간을 알려줘.
반드시 JSON 형식으로만 답변하고, 앞뒤에 인사말이나 부연 설명을 절대 붙이지 마.
{format_instructions}
"""

template2 = """
한국(인천 공항)에서 {country}까지 {flight_time}시간 걸린다.
비용(원화)과 기내 준비물을 5문장으로 설명하라.
앞뒤에 인사말이나 부연 설명을 절대 붙이지 마.
"""

prompt_template1 = PromptTemplate.from_template(
    template1,
    partial_variables={"format_instructions": parser.get_format_instructions()}
)
prompt_template2 = PromptTemplate.from_template(template2)

# 4. 체인 정의
chain1 = prompt_template1 | llm | parser              # 결과: FlightInfo 객체
chain2 = prompt_template2 | llm | StrOutputParser()   # 결과: 문자열

# 5. 합치기 — chain1 결과에서 flight_time 값만 추출해서 dict에 추가
combined = (
    RunnablePassthrough.assign(
        flight_time=chain1 | (lambda x: x.flight_time)
    )
    | chain2
)

# 6. 실행
result = combined.invoke({"country": "그린란드"})
print(result)